In [ ]:
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += os.pathsep + r"C:\hadoop\bin"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Olist_Analytics")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

print(spark)

In [ ]:
analytics_path = r"C:\MY ALL\Analytics Study\Project\Ecommerce Data Pipeline\Data\analytics_ready\spark\olist_item_analysis"

item_analysis = spark.read.parquet(analytics_path)

print("Rows:", item_analysis.count())
print("Columns:", len(item_analysis.columns))

Rows: 112650
Columns: 58


## Q1. Revenue & Sales Trend

*Business Question:*
How has the company's sales performance changed over time?

*Metrics:*
- Total Revenue
- Total Orders
- Items Sold
- Average Order Value (AOV)

*Approach:*
Aggregate order-level sales metrics by year and month to identify changes in sales performance over time.

In [ ]:
from pyspark.sql.functions import sum, countDistinct, avg

monthly_sales = (
    item_analysis
    .select(
        "order_id",
        "order_year",
        "order_month",
        "order_total_value"
    )
    .dropDuplicates(["order_id"])
    .groupBy("order_year", "order_month")
    .agg(
        sum("order_total_value").alias("total_revenue"),
        countDistinct("order_id").alias("total_orders")
    )
    .orderBy("order_year", "order_month")
)

monthly_sales.show(30, truncate=False)

+----------+-----------+------------------+------------+
|order_year|order_month|total_revenue     |total_orders|
+----------+-----------+------------------+------------+
|2016      |9          |354.75            |3           |
|2016      |10         |56808.84000000001 |308         |
|2016      |12         |19.62             |1           |
|2017      |1          |137188.49000000005|789         |
|2017      |2          |286280.6199999999 |1733        |
|2017      |3          |432048.58999999997|2641        |
|2017      |4          |412422.23999999993|2391        |
|2017      |5          |586190.9499999997 |3660        |
|2017      |6          |502963.03999999986|3217        |
|2017      |7          |584971.6199999998 |3969        |
|2017      |8          |668204.5999999994 |4293        |
|2017      |9          |720398.9100000005 |4243        |
|2017      |10         |769312.3700000001 |4568        |
|2017      |11         |1179143.7700000014|7451        |
|2017      |12         |863547.

In [ ]:
from pyspark.sql.functions import sum, countDistinct, round

monthly_items = (
    item_analysis
    .groupBy("order_year", "order_month")
    .agg(
        sum("order_item_count").alias("items_sold")
    )
)

monthly_sales = (
    monthly_sales
    .join(
        monthly_items,
        on=["order_year", "order_month"],
        how="left"
    )
    .withColumn(
        "avg_order_value",
        round(
            monthly_sales["total_revenue"] / monthly_sales["total_orders"],
            2
        )
    )
    .orderBy("order_year", "order_month")
)

monthly_sales.show(30, truncate=False)

+----------+-----------+------------------+------------+----------+---------------+
|order_year|order_month|total_revenue     |total_orders|items_sold|avg_order_value|
+----------+-----------+------------------+------------+----------+---------------+
|2016      |9          |354.75            |3           |14        |118.25         |
|2016      |10         |56808.84000000001 |308         |551       |184.44         |
|2016      |12         |19.62             |1           |1         |19.62          |
|2017      |1          |137188.49000000005|789         |1681      |173.88         |
|2017      |2          |286280.6199999999 |1733        |2553      |165.19         |
|2017      |3          |432048.58999999997|2641        |4030      |163.59         |
|2017      |4          |412422.23999999993|2391        |3504      |172.49         |
|2017      |5          |586190.9499999997 |3660        |5468      |160.16         |
|2017      |6          |502963.03999999986|3217        |4605      |156.35   

In [ ]:
monthly_sales = monthly_sales.withColumn(
    "total_revenue",
    round("total_revenue", 2)
)

monthly_sales.show(30, truncate=False)

+----------+-----------+-------------+------------+----------+---------------+
|order_year|order_month|total_revenue|total_orders|items_sold|avg_order_value|
+----------+-----------+-------------+------------+----------+---------------+
|2016      |9          |354.75       |3           |14        |118.25         |
|2016      |10         |56808.84     |308         |551       |184.44         |
|2016      |12         |19.62        |1           |1         |19.62          |
|2017      |1          |137188.49    |789         |1681      |173.88         |
|2017      |2          |286280.62    |1733        |2553      |165.19         |
|2017      |3          |432048.59    |2641        |4030      |163.59         |
|2017      |4          |412422.24    |2391        |3504      |172.49         |
|2017      |5          |586190.95    |3660        |5468      |160.16         |
|2017      |6          |502963.04    |3217        |4605      |156.35         |
|2017      |7          |584971.62    |3969        |6

In [ ]:
q1_output_path = r"C:\MY ALL\Analytics Study\Project\Ecommerce Data Pipeline\Analytics\Analytical_Tables\monthly_sales"

monthly_sales.write.mode("overwrite").parquet(q1_output_path)

print("Q1 analytical table saved successfully!")

Q1 analytical table saved successfully!


## Q2. Product Category Performance

*Business Question:*  
Which product categories generate the highest revenue and order volume?

*Metrics:*
- Total Revenue
- Total Orders
- Items Sold
- Average Order Value (AOV)

*Approach:*
Aggregate item-level sales by product category and count distinct orders to avoid double-counting.

In [ ]:
from pyspark.sql.functions import column, sum, countDistinct, round

category_performance = (
    item_analysis
    .groupBy("product_category_name_english")
    .agg(
        sum("item_total").alias("total_revenue"),
        countDistinct("order_id").alias("total_orders"),
        sum("order_item_count").alias("items_sold")
    )
    .withColumn(
        "avg_order_value",
        round(
            column("total_revenue") / column("total_orders"),
            2
        )
    )
    .orderBy("total_revenue", ascending=False)
)

category_performance.show(20, truncate=False)

+-----------------------------+------------------+------------+----------+---------------+
|product_category_name_english|total_revenue     |total_orders|items_sold|avg_order_value|
+-----------------------------+------------------+------------+----------+---------------+
|health_beauty                |1441248.0699999994|8836        |12479     |163.11         |
|watches_gifts                |1305541.6100000003|5624        |7195      |232.14         |
|bed_bath_table               |1241681.7199999995|9417        |16197     |131.86         |
|sports_leisure               |1156656.4799999997|7720        |11216     |149.83         |
|computers_accessories        |1059272.4000000006|6689        |11910     |158.36         |
|furniture_decor              |902511.7899999997 |6449        |14779     |139.95         |
|housewares                   |778397.7700000003 |5884        |11142     |132.29         |
|cool_stuff                   |719329.9499999998 |3632        |4369      |198.05         |

In [ ]:
category_performance.filter(
    column("product_category_name_english").isNull()
).show()

+-----------------------------+-----------------+------------+----------+---------------+
|product_category_name_english|    total_revenue|total_orders|items_sold|avg_order_value|
+-----------------------------+-----------------+------------+----------+---------------+
|                         NULL|213662.9000000003|        1473|      2193|         145.05|
+-----------------------------+-----------------+------------+----------+---------------+



In [ ]:
category_performance = category_performance.fillna(
    {"product_category_name_english": "Unknown"}
)

category_performance = category_performance.withColumn(
    "total_revenue",
    round("total_revenue", 2)
)

category_performance.show(20, truncate=False)

+-----------------------------+-------------+------------+----------+---------------+
|product_category_name_english|total_revenue|total_orders|items_sold|avg_order_value|
+-----------------------------+-------------+------------+----------+---------------+
|health_beauty                |1441248.07   |8836        |12479     |163.11         |
|watches_gifts                |1305541.61   |5624        |7195      |232.14         |
|bed_bath_table               |1241681.72   |9417        |16197     |131.86         |
|sports_leisure               |1156656.48   |7720        |11216     |149.83         |
|computers_accessories        |1059272.4    |6689        |11910     |158.36         |
|furniture_decor              |902511.79    |6449        |14779     |139.95         |
|housewares                   |778397.77    |5884        |11142     |132.29         |
|cool_stuff                   |719329.95    |3632        |4369      |198.05         |
|auto                         |685384.32    |3897     

In [ ]:
print("Number of categories:", category_performance.count())

category_performance.select(
    "product_category_name_english",
    "total_revenue",
    "total_orders",
    "items_sold",
    "avg_order_value"
).show(10, truncate=False)

Number of categories: 72
+-----------------------------+-------------+------------+----------+---------------+
|product_category_name_english|total_revenue|total_orders|items_sold|avg_order_value|
+-----------------------------+-------------+------------+----------+---------------+
|health_beauty                |1441248.07   |8836        |12479     |163.11         |
|watches_gifts                |1305541.61   |5624        |7195      |232.14         |
|bed_bath_table               |1241681.72   |9417        |16197     |131.86         |
|sports_leisure               |1156656.48   |7720        |11216     |149.83         |
|computers_accessories        |1059272.4    |6689        |11910     |158.36         |
|furniture_decor              |902511.79    |6449        |14779     |139.95         |
|housewares                   |778397.77    |5884        |11142     |132.29         |
|cool_stuff                   |719329.95    |3632        |4369      |198.05         |
|auto                        

In [ ]:
q2_output_path = r"C:\MY ALL\Analytics Study\Project\Ecommerce Data Pipeline\Analytics\Analytical_Tables\category_performance"

category_performance.write.mode("overwrite").parquet(q2_output_path)

print("Q2 analytical table saved successfully!")

Q2 analytical table saved successfully!


## Q3. Seller Performance

**Business Question:**  
Which sellers contribute the most to overall sales, and how concentrated is seller performance?

**Metrics:**  
- Total Sales
- Total Orders
- Items Sold
- Average Order Value (AOV)

**Approach:**  
Aggregate sales and order-level metrics by seller to compare seller performance.

In [ ]:
seller_performance = (
    item_analysis
    .select(
        "seller_id",
        "seller_item_count",
        "seller_order_count",
        "seller_total_sales",
        "seller_avg_order_value"
    )
    .dropDuplicates(["seller_id"])
    .orderBy("seller_total_sales", ascending=False)
)

seller_performance.show(20, truncate=False)

+--------------------------------+-----------------+------------------+------------------+----------------------+
|seller_id                       |seller_item_count|seller_order_count|seller_total_sales|seller_avg_order_value|
+--------------------------------+-----------------+------------------+------------------+----------------------+
|4869f7a5dfa277a7dca6462dcf3b52b2|1156             |1132              |249640.69999999995|220.53065371024732    |
|7c67e1448b00f6e969d365cea6b010ab|1364             |982               |239536.44000000006|243.92712830957237    |
|53243585a1d6dc2643021fd1853d8905|410              |358               |235856.68         |658.8175418994413     |
|4a3ca9315b744ce9f8e9374361493884|1987             |1806              |235539.95999999973|130.42079734219254    |
|fa1c13f2614d7b5c4749cbc52fecda94|586              |585               |204084.72999999998|348.86278632478627    |
|da8622b14eb17ae2831f4ac5b9dab84a|1551             |1314              |185192.3200000001

In [ ]:
seller_performance = (
    seller_performance
    .join(
        item_analysis
        .select("seller_id", "seller_city", "seller_state")
        .dropDuplicates(["seller_id"]),
        on="seller_id",
        how="left"
    )
    .orderBy("seller_total_sales", ascending=False)
)

seller_performance.show(20, truncate=False)

+--------------------------------+-----------------+------------------+------------------+----------------------+---------------------+------------+
|seller_id                       |seller_item_count|seller_order_count|seller_total_sales|seller_avg_order_value|seller_city          |seller_state|
+--------------------------------+-----------------+------------------+------------------+----------------------+---------------------+------------+
|4869f7a5dfa277a7dca6462dcf3b52b2|1156             |1132              |249640.69999999995|220.53065371024732    |guariba              |SP          |
|7c67e1448b00f6e969d365cea6b010ab|1364             |982               |239536.44000000006|243.92712830957237    |itaquaquecetuba      |SP          |
|53243585a1d6dc2643021fd1853d8905|410              |358               |235856.68         |658.8175418994413     |lauro de freitas     |BA          |
|4a3ca9315b744ce9f8e9374361493884|1987             |1806              |235539.95999999973|130.420797342192

In [ ]:
seller_performance = (
    seller_performance
    .withColumn("seller_total_sales", round("seller_total_sales", 2))
    .withColumn("seller_avg_order_value", round("seller_avg_order_value", 2))
)

seller_performance.show(20, truncate=False)

+--------------------------------+-----------------+------------------+------------------+----------------------+---------------------+------------+
|seller_id                       |seller_item_count|seller_order_count|seller_total_sales|seller_avg_order_value|seller_city          |seller_state|
+--------------------------------+-----------------+------------------+------------------+----------------------+---------------------+------------+
|4869f7a5dfa277a7dca6462dcf3b52b2|1156             |1132              |249640.7          |220.53                |guariba              |SP          |
|7c67e1448b00f6e969d365cea6b010ab|1364             |982               |239536.44         |243.93                |itaquaquecetuba      |SP          |
|53243585a1d6dc2643021fd1853d8905|410              |358               |235856.68         |658.82                |lauro de freitas     |BA          |
|4a3ca9315b744ce9f8e9374361493884|1987             |1806              |235539.96         |130.42          

In [ ]:
print("Number of sellers:", seller_performance.count())

print(
    "Total seller sales:",
    seller_performance.select(round(sum("seller_total_sales"), 2)).first()[0]
)

print(
    "Total seller orders:",
    seller_performance.select(sum("seller_order_count")).first()[0]
)

print(
    "Total seller items:",
    seller_performance.select(sum("seller_item_count")).first()[0]
)

Number of sellers: 3095
Total seller sales: 15843553.24
Total seller orders: 100010
Total seller items: 112650


In [ ]:
q3_output_path = r"C:\MY ALL\Analytics Study\Project\Ecommerce Data Pipeline\Analytics\Analytical_Tables\seller_performance"

seller_performance.write.mode("overwrite").parquet(q3_output_path)

print("Q3 analytical table saved successfully!")

Q3 analytical table saved successfully!


## Q4. Customer Behavior

**Business Question:**  
How much business comes from repeat customers versus one-time customers?

**Metrics:**  
- Total Customers
- One-Time Customers
- Repeat Customers
- Revenue from One-Time Customers
- Revenue from Repeat Customers
- Average Customer Spend

**Approach:**  
Segment customers based on their order frequency and compare their contribution to overall revenue.

In [ ]:
from pyspark.sql.functions import countDistinct, sum, avg, round, when, col

customer_behavior = (
    item_analysis
    .groupBy("customer_unique_id")
    .agg(
        countDistinct("order_id").alias("customer_order_count"),
        round(sum("item_total"), 2).alias("customer_total_spend")
    )
    .withColumn(
        "customer_type",
        when(col("customer_order_count") > 1, "Repeat")
        .otherwise("One-Time")
    )
)

customer_behavior.show(20, truncate=False)

+--------------------------------+--------------------+--------------------+-------------+
|customer_unique_id              |customer_order_count|customer_total_spend|customer_type|
+--------------------------------+--------------------+--------------------+-------------+
|7462a753a77ed093330660f7003064c7|2                   |145.72              |Repeat       |
|4a055e75b893bfa3d72f550bc325cf76|1                   |233.83              |One-Time     |
|a6408b4025cdea6e80ebbc09aeb59dde|1                   |97.77               |One-Time     |
|8033f408bcb75134075db3eb60b49937|1                   |130.0               |One-Time     |
|aa1228340b6d43ebc5f294f508c4a164|1                   |138.72              |One-Time     |
|1a3d1e8b28c0ee27c23333ee108c47e6|1                   |1924.03             |One-Time     |
|4011b7579e894fa929d2de97b4928f41|2                   |888.5               |Repeat       |
|f8d62e8ff724bf75c20ffd5faf7b9e30|1                   |164.58              |One-Time     |

In [ ]:
customer_summary = (
    customer_behavior
    .groupBy("customer_type")
    .agg(
        countDistinct("customer_unique_id").alias("total_customers"),
        round(sum("customer_total_spend"), 2).alias("total_revenue"),
        round(avg("customer_total_spend"), 2).alias("avg_customer_spend")
    )
    .orderBy("customer_type")
)

customer_summary.show(truncate=False)

+-------------+---------------+-------------+------------------+
|customer_type|total_customers|total_revenue|avg_customer_spend|
+-------------+---------------+-------------+------------------+
|One-Time     |92507          |1.493910699E7|161.49            |
|Repeat       |2913           |904446.25    |310.49            |
+-------------+---------------+-------------+------------------+



In [ ]:
customer_summary.select(
    sum("total_customers").alias("total_customers")
).show()

+---------------+
|total_customers|
+---------------+
|          95420|
+---------------+



In [ ]:
customer_summary.show(5)

In [ ]:
q4_output_path = r"C:\MY ALL\Analytics Study\Project\Ecommerce Data Pipeline\Analytics\Analytical_Tables\customer_behavior"

customer_summary.write.mode("overwrite").parquet(q4_output_path)

print("Q4 analytical table saved successfully!")

Q4 analytical table saved successfully!


## Q5. Geographic Performance

**Business Question:**  
Which customer states generate the most revenue and orders?

**Metrics:**  
- Total Revenue
- Total Orders
- Total Customers

**Approach:**  
Aggregate customer-level sales and order activity by customer state to compare geographic performance.

In [ ]:
from pyspark.sql.functions import sum, countDistinct, round

geographic_performance = (
    item_analysis
    .groupBy("customer_state")
    .agg(
        round(sum("item_total"), 2).alias("total_revenue"),
        countDistinct("order_id").alias("total_orders"),
        countDistinct("customer_unique_id").alias("total_customers")
    )
    .orderBy("total_revenue", ascending=False)
)

geographic_performance.show(20, truncate=False)

+--------------+-------------+------------+---------------+
|customer_state|total_revenue|total_orders|total_customers|
+--------------+-------------+------------+---------------+
|SP            |5921678.12   |41375       |39981          |
|RJ            |2129681.98   |12762       |12303          |
|MG            |1856161.49   |11544       |11178          |
|RS            |885826.76    |5432        |5249           |
|PR            |800935.44    |4998        |4840           |
|BA            |611506.67    |3358        |3257           |
|SC            |610213.6     |3612        |3513           |
|DF            |353229.44    |2125        |2062           |
|GO            |347706.93    |2007        |1942           |
|ES            |324801.91    |2025        |1956           |
|PE            |322237.69    |1648        |1605           |
|CE            |275606.3     |1327        |1305           |
|PA            |217647.11    |970         |944            |
|MT            |186168.96    |903       

In [ ]:
geographic_performance.select(
    sum("total_revenue").alias("total_revenue"),
    sum("total_orders").alias("total_orders"),
    sum("total_customers").alias("total_customers")
).show()

+--------------------+------------+---------------+
|       total_revenue|total_orders|total_customers|
+--------------------+------------+---------------+
|1.5843553239999998E7|       98666|          95458|
+--------------------+------------+---------------+



In [ ]:
q5_output_path = r"C:\MY ALL\Analytics Study\Project\Ecommerce Data Pipeline\Analytics\Analytical_Tables\geographic_performance"

geographic_performance.write.mode("overwrite").parquet(q5_output_path)

print("Q5 analytical table saved successfully!")

Q5 analytical table saved successfully!


## Q6. Delivery Performance

**Business Question:**  
How frequently are orders delivered late, and how does delivery performance vary?

**Metrics:**  
- Total Orders
- Late Orders
- Late Order %
- Average Delivery Days
- Average Delivery Delay

**Approach:**  
Analyze delivery performance at the order level and compare late-delivery rates across customer states.

In [ ]:
from pyspark.sql.functions import col

order_delivery = (
    item_analysis
    .select(
        "order_id",
        "customer_state",
        "delivery_days",
        "delivery_delay_days",
        "is_delivered_late",
        "delivery_status"
    )
    .dropDuplicates(["order_id"])
)

order_delivery.show(10, truncate=False)

+--------------------------------+--------------+-------------+-------------------+-----------------+---------------+
|order_id                        |customer_state|delivery_days|delivery_delay_days|is_delivered_late|delivery_status|
+--------------------------------+--------------+-------------+-------------------+-----------------+---------------+
|00018f77f2f0320c557190d7a144bdd3|SP            |16           |-3                 |0                |Early          |
|00042b26cf59d7ce69dfabb4e55b4fd9|SP            |25           |-16                |0                |Early          |
|00054e8431b9d7675808bcb819fb4a32|SP            |8            |-17                |0                |Early          |
|0006ec9db01a64e59a68b2c340bf65a7|RJ            |7            |-22                |0                |Early          |
|000aed2e25dbad2f9ddb70584c5a2ded|SP            |7            |-4                 |0                |Early          |
|00143d0f86d6fbd9f9b38ab440ac16f5|GO            |11     

In [ ]:
from pyspark.sql.functions import count, sum, avg, round

delivery_summary = (
    order_delivery
    .agg(
        count("order_id").alias("total_orders"),
        sum("is_delivered_late").alias("late_orders"),
        round(avg("delivery_days"), 2).alias("avg_delivery_days"),
        round(avg("delivery_delay_days"), 2).alias("avg_delivery_delay")
    )
    .withColumn(
        "late_order_percentage",
        round(
            col("late_orders") / col("total_orders") * 100,
            2
        )
    )
)

delivery_summary.show()

+------------+-----------+-----------------+------------------+---------------------+
|total_orders|late_orders|avg_delivery_days|avg_delivery_delay|late_order_percentage|
+------------+-----------+-----------------+------------------+---------------------+
|       98666|       6535|             12.5|            -11.88|                 6.62|
+------------+-----------+-----------------+------------------+---------------------+



In [ ]:
delivery_by_state = (
    order_delivery
    .groupBy("customer_state")
    .agg(
        count("order_id").alias("total_orders"),
        sum("is_delivered_late").alias("late_orders"),
        round(avg("delivery_days"), 2).alias("avg_delivery_days"),
        round(avg("delivery_delay_days"), 2).alias("avg_delivery_delay")
    )
    .withColumn(
        "late_order_percentage",
        round(
            col("late_orders") / col("total_orders") * 100,
            2
        )
    )
    .orderBy("late_order_percentage", ascending=False)
)

delivery_by_state.show(30, truncate=False)

+--------------+------------+-----------+-----------------+------------------+---------------------+
|customer_state|total_orders|late_orders|avg_delivery_days|avg_delivery_delay|late_order_percentage|
+--------------+------------+-----------+-----------------+------------------+---------------------+
|AL            |411         |85         |24.5             |-8.71             |20.68                |
|MA            |740         |125        |21.51            |-9.57             |16.89                |
|SE            |345         |51         |21.46            |-10.02            |14.78                |
|PI            |493         |66         |19.39            |-11.31            |13.39                |
|CE            |1327        |176        |21.2             |-10.8             |13.26                |
|BA            |3358        |396        |19.28            |-10.79            |11.79                |
|RJ            |12762       |1495       |15.24            |-11.77            |11.71        

In [ ]:
q6_output_path = r"C:\MY ALL\Analytics Study\Project\Ecommerce Data Pipeline\Analytics\Analytical_Tables\delivery_performance"

delivery_by_state.write.mode("overwrite").parquet(q6_output_path)

print("Q6 analytical table saved successfully!")

Q6 analytical table saved successfully!


## Q7. Customer Satisfaction

**Business Question:**  
Is delivery performance associated with customer review scores?

**Metrics:**  
- Average Review Score
- Total Reviews
- Average Delivery Delay
- Late Order %

**Approach:**  
Compare customer review scores across delivery-performance groups to understand whether late deliveries are associated with lower customer satisfaction.

In [ ]:
from pyspark.sql.functions import count, avg, round

delivery_satisfaction = (
    item_analysis
    .select(
        "order_id",
        "delivery_status",
        "delivery_delay_days",
        "review_score_avg"
    )
    .dropDuplicates(["order_id"])
    .groupBy("delivery_status")
    .agg(
        count("order_id").alias("total_orders"),
        round(avg("review_score_avg"), 2).alias("avg_review_score"),
        round(avg("delivery_delay_days"), 2).alias("avg_delivery_delay")
    )
    .orderBy("avg_review_score")
)

delivery_satisfaction.show(truncate=False)

+---------------+------------+----------------+------------------+
|delivery_status|total_orders|avg_review_score|avg_delivery_delay|
+---------------+------------+----------------+------------------+
|Not Delivered  |2190        |1.76            |NULL              |
|Late           |6535        |2.27            |10.62             |
|On Time        |1292        |4.04            |0.0               |
|Early          |88649       |4.29            |-13.71            |
+---------------+------------+----------------+------------------+



In [ ]:
q7_output_path = r"C:\MY ALL\Analytics Study\Project\Ecommerce Data Pipeline\Analytics\Analytical_Tables\delivery_satisfaction"

delivery_satisfaction.write.mode("overwrite").parquet(q7_output_path)

print("Q7 analytical table saved successfully!")

Q7 analytical table saved successfully!


## Q8. Freight & Order Economics

**Business Question:**  
How significant is freight cost relative to product price and order value?

**Metrics:**  
- Total Revenue
- Total Freight
- Average Freight Percentage
- Average Order Value
- Average Freight per Order

**Approach:**  
Analyze freight costs relative to product prices and order value to understand their impact on overall order economics.

In [ ]:
from pyspark.sql.functions import sum, avg, count, round, col

order_economics = (
    item_analysis
    .select(
        "order_id",
        "order_total_value",
        "order_total_freight",
        "freight_percentage"
    )
    .dropDuplicates(["order_id"])
    .agg(
        round(sum("order_total_value"), 2).alias("total_revenue"),
        round(sum("order_total_freight"), 2).alias("total_freight"),
        round(avg("freight_percentage"), 2).alias("avg_freight_percentage"),
        round(avg("order_total_value"), 2).alias("avg_order_value"),
        round(avg("order_total_freight"), 2).alias("avg_freight_per_order")
    )
)

order_economics.show(truncate=False)

+-------------+-------------+----------------------+---------------+---------------------+
|total_revenue|total_freight|avg_freight_percentage|avg_order_value|avg_freight_per_order|
+-------------+-------------+----------------------+---------------+---------------------+
|1.584355324E7|2251909.54   |30.91                 |160.58         |22.82                |
+-------------+-------------+----------------------+---------------+---------------------+



In [ ]:
q8_output_path = r"C:\MY ALL\Analytics Study\Project\Ecommerce Data Pipeline\Analytics\Analytical_Tables\order_economics"

order_economics.write.mode("overwrite").parquet(q8_output_path)

print("Q8 analytical table saved successfully!")

Q8 analytical table saved successfully!
